# Creating new Queries

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM
logging.set_verbosity_error()
import numpy as np
import os
from torch import Tensor
from torch.utils.data import DataLoader
import faiss
import json
from beir.datasets.data_loader import GenericDataLoader
from tqdm import tqdm
from dotenv import load_dotenv
import os
import re

/work/mbouthil/.conda/envs/myuwenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Loading Data
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841823/8841823 [01:17<00:00, 114566.00it/s]


Gathering Unique Passages

In [129]:
count = 1

pass_info = []

for key in qrels.keys():
    if len(qrels[key]) < 2:
        pass_info.append((key, list(qrels[key].keys())[0], list(qrels[key].values())[0]))

In [5]:
pass_info = [(key, list(qrels[key].keys())[0], list(qrels[key].values())[0]) 
             for key in qrels.keys() if len(qrels[key]) < 2]

In [6]:
print(pass_info[:10])

[('1185869', '0', 1), ('1185868', '16', 1), ('597651', '49', 1), ('403613', '60', 1), ('1183785', '389', 1), ('312651', '616', 1), ('80385', '723', 1), ('645590', '944', 1), ('645337', '1054', 1), ('186154', '1160', 1)]


In [133]:
unique_passages = [corpus[str(id)]['text'] for _, id, _ in pass_info]
unique_passages[:2]

['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.',
 'The approach is based on a theory of justice that considers crime and wrongdoing to be an offense against an individual or community, rather than the State. Restorative justice that fosters dialogue between victim and offender has shown the highest rates of victim satisfaction and offender accountability.']

# Initializing LLM

In [17]:
# Authenticating Token
load_dotenv('/work/mbouthil/MMATH-CM-Research-Project/token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model and Tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token
)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 366.09it/s, Materializing param=model.norm.weight]                              


In [31]:
system_prompt='''
You are a helful AI assistant. You are to write a query for the provided passage. Provide only the new query.
'''

In [19]:
### LLM function ###
def llm_pass(
        messages:list[list[dict]],
        padding:bool=True,
        truncation:bool=True,
        max_tokens:int=256, 
        temp:float=0.1,
        top_p:float=0.9,
) -> list[str]:

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=padding,
        truncation=truncation
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temp,
            top_p=top_p,
            do_sample=True
        )

    responses = []
    for i in range(len(messages)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

In [23]:
messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": passage}
        ]
        for passage in unique_passages[:2]
    ]

In [9]:
def batch_splits(item:list, batch_size:int=64) -> list[tuple[str, str]]:

    for i in range(0, len(item), batch_size):
        yield item[i:i + batch_size]

batches = batch_splits(pass_info)

In [10]:
for batch in batches:
    print(batch)
    break

[('1185869', '0', 1), ('1185868', '16', 1), ('597651', '49', 1), ('403613', '60', 1), ('1183785', '389', 1), ('312651', '616', 1), ('80385', '723', 1), ('645590', '944', 1), ('645337', '1054', 1), ('186154', '1160', 1), ('457407', '1172', 1), ('441383', '1389', 1), ('683408', '1605', 1), ('1164799', '1713', 1), ('484187', '1822', 1), ('460668', '1939', 1), ('666321', '2152', 1), ('182487', '2277', 1), ('564233', '2488', 1), ('455279', '2599', 1), ('208108', '2704', 1), ('733739', '2816', 1), ('1164798', '2924', 1), ('402608', '3036', 1), ('443797', '3146', 1), ('662502', '3257', 1), ('1184679', '3368', 1), ('14562', '3382', 1), ('602162', '3597', 1), ('545059', '3702', 1), ('708236', '3815', 1), ('310130', '3923', 1), ('693161', '4029', 1), ('186617', '4251', 1), ('573027', '4360', 1), ('1173772', '4462', 1), ('541973', '4583', 1), ('273090', '4698', 1), ('441269', '4809', 1), ('642237', '4918', 1), ('503515', '5025', 1), ('637443', '5250', 1), ('1164796', '5359', 1), ('135841', '5585'

In [121]:
new_passages = llm_pass(messages)
new_passages = [passage[11:] for passage in new_passages]

In [131]:
print(new_passages[0])

What was the significance of communication among scientists in the Manhattan Project, and what were the consequences of their achievement?


### Creating a mapping

In [48]:
max_id = max([int(key) for key in corpus.keys()])

In [57]:
test = {**{str(max_id+1): 1}, **scores[0]}
print(test)

{'8841823': 1, '0': 1}


In [59]:
test['0']

1

In [62]:
qrels_test = qrels.copy()

In [148]:
for i, tuple in enumerate(pass_info):
    if i == 2:
        break
    q_id, p_id, score = tuple
    qrels_test[q_id] = {p_id: score, str(max_id+i): score}
    corpus[max_id+i] = {'text': new_passages[i], 'title': ''}
    print(corpus[max_id+i])

{'text': 'What was the significance of communication among scientists in the Manhattan Project, and what were the consequences of their achievement?', 'title': ''}
{'text': "Here's a possible query based on the passage:\n\nWhat is the underlying theory of justice that views crime as an offense against an individual or community rather than the State, and what are the benefits of restorative justice in this context?", 'title': ''}


In [147]:
print(corpus['1'])

{'text': 'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.', 'title': ''}


In [143]:
print(qrels['3'])

{'1142680': 1}


In [91]:
print(qrels['3'])

{'1142680': 1}


In [92]:
print(qrels_test['3'])

{'1142680': 1, '8904029': 1}


In [94]:
print(corpus['1142680']['text'])

The primary (parts of the cortex that receive sensory inputs from the thalamus) visual cortex is also known as V1, V isual area one, and the striate cortex.The extrastriate areas consist of visual areas two (V2), three (V3), four (V4), and five (V5).he primary visual cortex is the best-studied visual area in the brain. In all mammals studied, it is located in the posterior pole of the occipital cortex (the occipital cortex is responsible for processing visual stimuli).


In [95]:
print(corpus['8904029'])

KeyError: '8904029'

In [13]:
def batch_splits(item:list, batch_size:int=64):

    for i in range(0, len(item), batch_size):
        yield item[i:i + batch_size]

batches = batch_splits(pass_info)

test = True

max_id = max([int(key) for key in corpus.keys()])


for batch in batches:

    max_id = max([int(key) for key in corpus.keys()])
    passages = [corpus[str(id)]['text'] for _, id, _ in batch]
    # messages = [
    #     [
    #         {"role": "system", "content": system_prompt},
    #         {"role": "user", "content": passage}
    #     ]
    #     for passage in passages
    # ]
    # new_passages = llm_pass(messages)
    new_passages = passages


    for i, tuple in enumerate(batch):
        q_id, p_id, score = tuple
        qrels[q_id] = {p_id: score, str(max_id+i): score}
        corpus[max_id+i] = {'text': new_passages[i], 'title': ''}

    if test == True:
        break